# Screencast 2 — Quantização de Modelos (PyTorch)

**Objetivo deste notebook:**

Carregar o **modelo de referência** (ResNet50 + transfer learning) treinado no Screencast 1 e explorar quatro variantes de quantização.

1. **FP32 em CPU (referência justa)** — os backends de quantização do PyTorch (`fbgemm`/`x86`, `qnnpack`) só rodam em CPU; medimos aqui a mesma linha de base, mas em CPU, para que a comparação com as variantes quantizadas seja justa.
2. **Quantização dinâmica pós-treinamento** — o caminho mais simples do PyTorch: só os pesos das camadas `Linear` são convertidos para INT8, as ativações são quantizadas "on the fly" durante a inferência.
3. **Quantização estática pós-treinamento (INT8)** — pesos **e** ativações em INT8, calibrados com um dataset representativo, usando *FX Graph Mode Quantization* (`torch.ao.quantization.quantize_fx`).
4. **Quantização ciente do treinamento (QAT)** — o modelo é re-treinado simulando os efeitos da quantização (via `prepare_qat_fx`), depois convertido para INT8.
5. **Comparação final** — tamanho, latência e acurácia de todas as variantes lado a lado, incluindo a linha de base do Screencast 1.


## 1. Ambiente e Carregamento do Modelo de Referência

Este notebook assume que o **Screencast 1 (`01_setup_pytorch.ipynb`)**, já foi executado e que os seguintes arquivos existem:

- `models/edge_ai_reference_resnet50_pytorch.pt`
- `models/baseline_metrics_pytorch.json`
- O dataset extraído em `data/` (mesma estrutura `train/`, `test/`).

In [ ]:
import os
import time
import json
import copy
import pathlib
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

print("PyTorch:", torch.__version__)

# Backends de quantização do PyTorch rodam apenas em CPU (eager e FX graph mode).
# "x86" é o backend recomendado para servidores/desktops x86 (sucessor do "fbgemm");
# para deploy em dispositivos ARM (ex.: Raspberry Pi, celulares), o equivalente é "qnnpack".
DEVICE_QUANT = torch.device("cpu")
torch.backends.quantized.engine = "x86"
print("Backend de quantização:", torch.backends.quantized.engine)

MODELS_DIR = pathlib.Path("../models")
DATA_DIR = pathlib.Path("../data")

REFERENCE_MODEL_PATH = MODELS_DIR / "edge_ai_reference_resnet50_pytorch.pt"
BASELINE_METRICS_PATH = MODELS_DIR / "baseline_metrics_pytorch.json"

assert REFERENCE_MODEL_PATH.exists(), (
    "Modelo de referencia (PyTorch) nao encontrado. Execute o Screencast 1, versao PyTorch "
    "(01_setup_pytorch.ipynb), primeiro."
)

with open(BASELINE_METRICS_PATH, "r", encoding="utf-8") as f:
    baseline_metrics = json.load(f)

print(json.dumps(baseline_metrics, indent=2, ensure_ascii=False))


In [ ]:
# weights_only=False: o Screencast 1 salvou o objeto do modelo inteiro (torch.save(model, path)),
# não apenas os pesos (state_dict) — precisamos desserializar a arquitetura completa também.
reference_model = torch.load(REFERENCE_MODEL_PATH, map_location="cpu", weights_only=False)
reference_model.eval()

IMG_SIZE = tuple(baseline_metrics["img_size"])
print(f"Modelo de referencia carregado. Entrada esperada: {(3,) + IMG_SIZE}")


## 2. Recarregando o Pipeline de Dados

Mesma configuração do Screencast 1 — precisamos do conjunto de teste (para avaliar acurácia), do split de validação (para o *early stopping* do fine-tuning de QAT) e de uma pequena amostra do conjunto de treino (para calibrar a quantização estática INT8 pós-treinamento e para o próprio fine-tuning da quantização ciente do treinamento).


In [ ]:
def find_split_dirs(root: pathlib.Path):
    train_dirs = list(root.rglob("train"))
    test_dirs = list(root.rglob("test"))
    train_dir = next((d for d in train_dirs if (d / "ok_front").exists()), None)
    test_dir = next((d for d in test_dirs if (d / "ok_front").exists()), None)
    return train_dir, test_dir

TRAIN_DIR, TEST_DIR = find_split_dirs(DATA_DIR)
assert TRAIN_DIR is not None and TEST_DIR is not None, "Dataset não encontrado em ./data — execute o Screencast 1 primeiro."

BATCH_SIZE = 32
SEED = 42
torch.manual_seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

data_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

CLASS_TO_IDX = {"ok_front": 0, "def_front": 1}

def remap_target(default_class_to_idx):
    inverse_default = {v: k for k, v in default_class_to_idx.items()}
    return lambda idx: CLASS_TO_IDX[inverse_default[idx]]

full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=data_transform)
full_train_dataset.target_transform = remap_target(full_train_dataset.class_to_idx)

test_dataset = datasets.ImageFolder(TEST_DIR, transform=data_transform)
test_dataset.target_transform = remap_target(test_dataset.class_to_idx)

n_val = int(len(full_train_dataset) * 0.2)
n_train = len(full_train_dataset) - n_val
train_dataset, val_dataset = random_split(
    full_train_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
# shuffle=True (com seed fixa) — importante: sem shuffle, ImageFolder agrupa os arquivos por
# classe, e qualquer corte por MAX_EVAL_SAMPLES (abaixo) acabaria avaliando quase só uma classe
# (ex.: só ok_front), enviesando a acurácia medida.
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Treino: {n_train} imagens | Validação: {n_val} imagens | Teste: {len(test_dataset)} imagens")


## 3. Funções Utilitárias — Tamanho, Latência e Acurácia

Estas funções vão ser reaproveitadas para **todas** as variantes de quantização, garantindo que a comparação final seja justa (mesma metodologia de medição para todas).


In [ ]:
def save_model(model, path: pathlib.Path):
    # Serializa o objeto do modelo inteiro (arquitetura + pesos), assim como no Screencast 1 —
    # necessário aqui porque modelos quantizados (dinâmicos ou FX GraphModule) têm classes
    # Python diferentes do modelo original; salvar só o state_dict não seria suficiente para
    # recarregar a arquitetura quantizada depois.
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model, path)
    return path.stat().st_size / (1024 * 1024)  # tamanho em MB


def measure_latency_ms(model, sample_batch, device=DEVICE_QUANT, n_runs=100):
    model.eval()
    sample_batch = sample_batch.to(device)
    with torch.no_grad():
        _ = model(sample_batch)  # warm-up
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(sample_batch)
            times.append((time.perf_counter() - start) * 1000)
    return float(np.mean(times)), float(np.std(times))


def evaluate_accuracy(model, loader, device=DEVICE_QUANT, max_samples=None):
    model.eval()
    y_true, y_pred = [], []
    seen = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            preds = (probs >= 0.5).astype(int)
            y_pred.extend(preds.tolist())
            y_true.extend(labels.numpy().ravel().tolist())
            seen += len(preds)
            if max_samples is not None and seen >= max_samples:
                break

    if max_samples is not None:
        y_true, y_pred = y_true[:max_samples], y_pred[:max_samples]

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return float((y_true == y_pred).mean())


def make_calibration_batches(loader, num_samples=100):
    # Gerador de lotes de calibração para a quantização estática INT8 pós-treinamento    
    seen = 0
    for images, _ in loader:
        yield images
        seen += images.shape[0]
        if seen >= num_samples:
            return


In [ ]:
# Amostra fixa (1 imagem) para medição de latência — mesma amostra usada em todas as variantes
sample_batch, _ = next(iter(test_loader))
LATENCY_SAMPLE = sample_batch[:1]

# Amostra de avaliação — use um valor menor (ex.: 200) para iterar mais rápido durante a gravação;
# use None para avaliar no conjunto de teste completo antes da entrega final.
MAX_EVAL_SAMPLES = None

results = {}  # nome_da_variante -> {"size_mb", "latency_ms_mean", "latency_ms_std", "accuracy"}

MODELS_DIR_QUANT = MODELS_DIR / "pytorch_quant"
MODELS_DIR_QUANT.mkdir(parents=True, exist_ok=True)


## 4. Variante 0 — FP32 em CPU (referência justa)

Os backends de quantização do PyTorch (`x86`/`fbgemm`, `qnnpack`) só têm kernels de inferência para **CPU** — não existe hoje um caminho oficial para rodar um modelo INT8 quantizado por essas rotas em GPU. Isso significa que, se o Screencast 1 foi executado em uma GPU (`DEVICE` = `cuda`), comparar a latência quantizada diretamente com a linha de base salva em `baseline_metrics_pytorch.json` seria injusto — a diferença misturaria o efeito da quantização com o efeito de trocar de hardware.

Por isso, antes de quantizar, medimos aqui a mesma linha de base **em CPU**, que serve como o ponto de comparação real para as três variantes quantizadas a seguir.


In [ ]:
reference_model_cpu = reference_model.to("cpu")
reference_model_cpu.eval()

size_mb = save_model(reference_model_cpu, MODELS_DIR_QUANT / "model_fp32_cpu.pt")
latency_mean, latency_std = measure_latency_ms(reference_model_cpu, LATENCY_SAMPLE)
accuracy = evaluate_accuracy(reference_model_cpu, test_loader, max_samples=MAX_EVAL_SAMPLES)

results["fp32_cpu"] = {
    "size_mb": size_mb, "latency_ms_mean": latency_mean,
    "latency_ms_std": latency_std, "accuracy": accuracy,
}
print(results["fp32_cpu"])


## 5. Variante 1 — Quantização Dinâmica Pós-Treinamento

A forma mais simples de quantização do PyTorch: os **pesos** das camadas suportadas (por padrão, `nn.Linear` e `nn.LSTM`) são convertidos para INT8 uma única vez; as **ativações** são quantizadas dinamicamente, em tempo real, a cada chamada de inferência — por isso não precisa de dataset de calibração nem de re-treinamento.

**Nota para o desenvolvedor:** a quantização dinâmica do PyTorch **não** cobre `nn.Conv2d` — e a ResNet50 é uma rede quase inteiramente convolucional, com apenas a cabeça de classificação (`fc`) em camadas `Linear`. Isso significa que o ganho de tamanho/latência aqui tende a ser pequeno (só a cabeça é afetada), diferente de um modelo dominado por camadas `Linear` (ex.: Transformers), onde a quantização dinâmica sozinha já traz ganhos expressivos. Para quantizar as convoluções — a maior parte dos parâmetros da ResNet50 — precisamos da quantização **estática** (Seção 6).


In [ ]:
dynamic_model = torch.ao.quantization.quantize_dynamic(
    copy.deepcopy(reference_model_cpu),
    {nn.Linear},
    dtype=torch.qint8,
)
dynamic_model.eval()

size_mb = save_model(dynamic_model, MODELS_DIR_QUANT / "model_dynamic_int8.pt")
latency_mean, latency_std = measure_latency_ms(dynamic_model, LATENCY_SAMPLE)
accuracy = evaluate_accuracy(dynamic_model, test_loader, max_samples=MAX_EVAL_SAMPLES)

results["dynamic_ptq"] = {
    "size_mb": size_mb, "latency_ms_mean": latency_mean,
    "latency_ms_std": latency_std, "accuracy": accuracy,
}
print(results["dynamic_ptq"])


## 6. Variante 2 — Quantização Estática Pós-Treinamento (INT8, FX Graph Mode)

Aqui pesos **e ativações** de praticamente todo o modelo (incluindo as convoluções) são convertidos para INT8. Isso exige um pequeno lote de dados representativos para **calibrar** os intervalos numéricos (min/max observados) de cada camada antes da conversão.

Usamos o *FX Graph Mode Quantization* (`torch.ao.quantization.quantize_fx`), a API recomendada pelo PyTorch para quantizar modelos "prontos" como a ResNet50: ele traça o grafo do modelo automaticamente, funde padrões como `Conv2d + BatchNorm + ReLU` em uma única operação quantizada, e insere os observadores de calibração — tudo sem precisar reescrever a arquitetura manualmente (diferente do *eager mode*, que exigiria fundir camadas à mão com `torch.ao.quantization.fuse_modules`).

In [ ]:
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

example_inputs = (LATENCY_SAMPLE,)

qconfig_mapping = get_default_qconfig_mapping("x86")

model_to_prepare = copy.deepcopy(reference_model_cpu).eval()
prepared_static = prepare_fx(model_to_prepare, qconfig_mapping, example_inputs)

# Calibração — passa uma pequena amostra representativa pelo modelo preparado; os observadores
# inseridos por prepare_fx registram min/max de ativações em cada camada durante essas chamadas.
prepared_static.eval()
with torch.no_grad():
    for calib_batch in make_calibration_batches(train_loader, num_samples=100):
        prepared_static(calib_batch)

static_model = convert_fx(prepared_static)
static_model.eval()

size_mb = save_model(static_model, MODELS_DIR_QUANT / "model_int8_static_ptq.pt")
latency_mean, latency_std = measure_latency_ms(static_model, LATENCY_SAMPLE)
accuracy = evaluate_accuracy(static_model, test_loader, max_samples=MAX_EVAL_SAMPLES)

results["int8_static_ptq"] = {
    "size_mb": size_mb, "latency_ms_mean": latency_mean,
    "latency_ms_std": latency_std, "accuracy": accuracy,
}
print(results["int8_static_ptq"])


In [ ]:
queda_acc_int8_ptq = results["fp32_cpu"]["accuracy"] - results["int8_static_ptq"]["accuracy"]
print(f"Queda de acurácia (FP32 em CPU -> INT8 estático pós-treinamento): {queda_acc_int8_ptq:.4f}")

if queda_acc_int8_ptq > 0.02:
    print(
        "\nQueda de acurácia relevante (> 2 pontos percentuais). "
        "Este é exatamente o cenário em que quantização ciente do treinamento (QAT) se justifica — "
        "vamos explorar isso na próxima seção."
    )
else:
    print(
        "\nQueda de acurácia pequena. Mesmo assim, vamos treinar a versão QAT a seguir "
        "para comparar as duas abordagens lado a lado."
    )


## 7. Variante 3 — Quantização INT8 Ciente do Treinamento (QAT)

Diferente da abordagem pós-treinamento, aqui o modelo é envolvido com "nós de fake-quantization" *antes* do fine-tuning, para que o processo de treinamento já aprenda a compensar o ruído numérico introduzido pela quantização — usando `prepare_qat_fx`, o equivalente em QAT do `prepare_fx` da seção anterior.

**Nota para o desenvolvedor:** o fine-tuning aqui roda inteiramente em CPU (mesma limitação de backend das seções 5 e 6), o que o torna mais lento do que o treinamento original em GPU do Screencast 1. Por isso usamos poucas épocas — o objetivo do QAT é adaptar o modelo ao ruído de quantização, não re-treinar do zero.


In [ ]:
from torch.ao.quantization import get_default_qat_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_qat_fx

qat_qconfig_mapping = get_default_qat_qconfig_mapping("x86")

model_to_qat = copy.deepcopy(reference_model_cpu).train()
qat_prepared = prepare_qat_fx(model_to_qat, qat_qconfig_mapping, example_inputs)
qat_prepared.to(DEVICE_QUANT)

print("Modelo anotado para QAT com sucesso (fake-quantization inserida via FX).")


In [ ]:
def run_epoch(model, loader, optimizer, criterion, training, device=DEVICE_QUANT):
    model.train(mode=training)
    total_loss, total_correct, total_samples = 0.0, 0, 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            if training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def fit(model, optimizer, train_loader, val_loader, epochs, patience=1, device=DEVICE_QUANT):
    criterion = nn.BCEWithLogitsLoss()
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, training=True, device=device)
        val_loss, val_acc = run_epoch(model, val_loader, optimizer, criterion, training=False, device=device)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch {epoch}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
            f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping na época {epoch} (sem melhora em {patience} épocas).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)  # restore_best_weights=True

    return history


qat_optimizer = torch.optim.Adam(qat_prepared.parameters(), lr=1e-5)
history_qat = fit(qat_prepared, qat_optimizer, train_loader, val_loader, epochs=2, patience=1)


In [ ]:
qat_prepared.eval()
qat_model = convert_fx(qat_prepared)
qat_model.eval()

size_mb = save_model(qat_model, MODELS_DIR_QUANT / "model_int8_qat.pt")
latency_mean, latency_std = measure_latency_ms(qat_model, LATENCY_SAMPLE)
accuracy = evaluate_accuracy(qat_model, test_loader, max_samples=MAX_EVAL_SAMPLES)

results["int8_qat"] = {
    "size_mb": size_mb, "latency_ms_mean": latency_mean,
    "latency_ms_std": latency_std, "accuracy": accuracy,
}
print(results["int8_qat"])


## 8. Comparação Final

Reunindo a linha de base original do Screencast 1 (no dispositivo em que foi treinada) com a linha de base em CPU e todas as variantes de quantização exploradas aqui.


In [ ]:
comparison = {
    "pytorch_fp32_baseline_original_device": {
        "size_mb": baseline_metrics["size_mb"],
        "latency_ms_mean": baseline_metrics["latency_ms_mean"],
        "latency_ms_std": baseline_metrics["latency_ms_std"],
        "accuracy": baseline_metrics["test_accuracy"],
    },
    **results,
}

print(f"{'Variante':<32}{'Tamanho (MB)':>14}{'Latência (ms)':>16}{'Acurácia':>12}")
for name, m in comparison.items():
    print(f"{name:<32}{m['size_mb']:>14.2f}{m['latency_ms_mean']:>16.2f}{m['accuracy']:>12.4f}")


In [ ]:
names = list(comparison.keys())
sizes = [comparison[n]["size_mb"] for n in names]
latencies = [comparison[n]["latency_ms_mean"] for n in names]
accuracies = [comparison[n]["accuracy"] for n in names]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(names, sizes, color="#4C72B0")
axes[0].set_title("Tamanho em disco (MB)")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(names, latencies, color="#DD8452")
axes[1].set_title("Latência média por inferência (ms)")
axes[1].tick_params(axis="x", rotation=30)

axes[2].bar(names, accuracies, color="#55A868")
axes[2].set_title("Acurácia no conjunto de teste")
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
COMPARISON_PATH = MODELS_DIR / "quantization_comparison_pytorch.json"
with open(COMPARISON_PATH, "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)
0
print(f"Comparação salva em: {COMPARISON_PATH.resolve()}")


## 9. Checklist e Próximos Passos

- [x] Modelo de referência do Screencast 1 carregado.
- [x] Linha de base FP32 remedida em CPU (referência justa para as variantes quantizadas).
- [x] Quantização dinâmica pós-treinamento (`torch.ao.quantization.quantize_dynamic`).
- [x] Quantização estática pós-treinamento INT8 (FX Graph Mode, com calibração).
- [x] Quantização ciente do treinamento (QAT, via `prepare_qat_fx`).
- [x] Comparação de tamanho, latência e acurácia entre todas as variantes, salva em `models/quantization_comparison_pytorch.json`.

**No Screencast 3 (Pruning)**,  vamos voltar ao mesmo `edge_ai_reference_resnet50_pytorch.pt` — agora aplicando poda de pesos (`torch.nn.utils.prune`, estruturada)
